# 02 — LSTM Fraud Detector: Architecture & Training (Days 3–4)

**Day 3:** Implement `LSTMFraudDetector`, run calibration on 20% subset (5 epochs).  
**Day 4:** Full training on 100% data (10 epochs), generate training curves, save checkpoints.

**Runtime:** Runtime → Change runtime type → **T4 GPU** before running anything.

## 0. GPU Check

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU.')

## 1. Install Dependencies

In [ ]:
!pip install -q torch scikit-learn numpy pandas matplotlib pyyaml kaggle

## 2. Clone Repo & Set Working Directory

Clones the `feature/lstm-model` branch — that's where the notebooks, model code, and config live.

In [ ]:
import os, shutil

REPO_URL = 'https://github.com/owenz4040/Meridaian.git'
BRANCH   = 'feature/lstm-model'
REPO_DIR = '/content/meridian-sentinel'

# Move to /content first so we don't delete our own CWD
os.chdir('/content')

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git branch
print('src/ exists:', os.path.exists('src'))

## 3. Download PaySim Dataset (Kaggle API)

**Real dataset:** `PS_20174392719_1491204439457_log.csv` — 6,362,621 rows, ~500 MB.

**Before running this cell:**
1. Go to [kaggle.com](https://www.kaggle.com) → Your profile → Settings → API → **Create New Token**
2. This downloads `kaggle.json` to your machine
3. Run the cell below — it will prompt you to upload that file

In [ ]:
from google.colab import files
import os

# Upload kaggle.json credentials
print('Upload your kaggle.json file when prompted...')
uploaded = files.upload()  # select kaggle.json from your Downloads folder

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle credentials configured.')

In [ ]:
os.makedirs('data', exist_ok=True)

CSV_NAME = 'PS_20174392719_1491204439457_log.csv'
CSV_PATH = f'data/{CSV_NAME}'

if os.path.exists(CSV_PATH):
    print(f'CSV already present: {CSV_PATH}')
else:
    print('Downloading PaySim from Kaggle (~500 MB) ...')
    !kaggle datasets download -d ealaxi/paysim1 -p data/ --unzip

# Verify row count — should be 6,362,621
!echo -n 'Row count (incl. header): ' && wc -l < {CSV_PATH}
!ls -lh {CSV_PATH}

## 4. Run the Feature Engineering Pipeline on Real PaySim Data

This calls `src/pipeline/run_pipeline.py --csv <path>` which:
- Loads all 6.3M rows
- SHA-256 hashes `nameOrig` / `nameDest` (PII obfuscation)
- Engineers the 12 features and builds sliding-window sequences of 5
- Splits 70/15/15 stratified on `isFraud`
- Saves `X_train.npy`, `y_train.npy`, etc. to `data/processed/`

Expected runtime: **5–10 min** on Colab CPU (pipeline is not GPU-bound).

In [ ]:
import sys, os

REPO_DIR = '/content/meridian-sentinel'
CSV_PATH = 'data/PS_20174392719_1491204439457_log.csv'

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Pull latest pipeline changes (adds --sample flag + float32 loading)
!git pull origin feature/lstm-model

# Flush any cached version of the pipeline modules
for _mod in [k for k in sys.modules if k.startswith('src.pipeline')]:
    del sys.modules[_mod]

print('cwd  :', os.getcwd())
print('src/ :', os.path.exists('src'))

sys.argv = ['run_pipeline', '--csv', CSV_PATH, '--sample', '2000000']
from src.pipeline.run_pipeline import main
main()

In [ ]:
import numpy as np

DATA_DIR = 'data/processed'
X_train = np.load(f'{DATA_DIR}/X_train.npy').astype(np.float32)
y_train = np.load(f'{DATA_DIR}/y_train.npy').astype(np.float32)
X_val   = np.load(f'{DATA_DIR}/X_val.npy').astype(np.float32)
y_val   = np.load(f'{DATA_DIR}/y_val.npy').astype(np.float32)
X_test  = np.load(f'{DATA_DIR}/X_test.npy').astype(np.float32)
y_test  = np.load(f'{DATA_DIR}/y_test.npy').astype(np.float32)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}  y_val: {y_val.shape}')
print(f'X_test:  {X_test.shape}  y_test: {y_test.shape}')
print(f'Train fraud ratio: {y_train.mean():.4%}  (expect ~0.13% for real PaySim)')

# Compute pos_weight dynamically from actual fraud ratio
fraud_rate = y_train.mean()
pos_weight_val = float((1.0 - fraud_rate) / fraud_rate) if fraud_rate > 0 else 1.0
print(f'Computed pos_weight: {pos_weight_val:.1f}  (expect ~769 for real PaySim)')

## 5. Load Config & Build Model

In [ ]:
import yaml

with open('config/model_config.yaml') as f:
    cfg = yaml.safe_load(f)

from src.models.lstm_model import build_model

model = build_model(cfg).to(device)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6. DataLoader Utilities

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=(device.type == 'cuda'))

BATCH_SIZE = cfg['training']['batch_size']
train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

## 7. Training Loop Utilities

In [ ]:
import torch.nn as nn

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(y_batch)
        preds = (torch.sigmoid(logits) >= 0.5).long()
        correct += (preds == y_batch.long()).sum().item()
        total += len(y_batch)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * len(y_batch)
            preds = (torch.sigmoid(logits) >= 0.5).long()
            correct += (preds == y_batch.long()).sum().item()
            total += len(y_batch)
    return total_loss / total, correct / total

---
# DAY 3 — Calibration Run (20% subset, 5 epochs)

In [ ]:
from sklearn.model_selection import train_test_split
import time, json

CALIB_SUBSET = cfg['training']['calibration_subset']  # 0.20
CALIB_EPOCHS = cfg['training']['calibration_epochs']  # 5
LR           = cfg['training']['learning_rate']
SEED         = cfg['training']['seed']

# Stratified 20% subset
idx = np.arange(len(X_train))
idx_calib, _ = train_test_split(
    idx, train_size=CALIB_SUBSET, stratify=y_train.astype(int), random_state=SEED
)
X_calib, y_calib = X_train[idx_calib], y_train[idx_calib]
calib_loader = make_loader(X_calib, y_calib, BATCH_SIZE, shuffle=True)
print(f'Calibration subset: {len(X_calib):,} samples  (fraud: {y_calib.mean():.4%})')

In [ ]:
torch.manual_seed(SEED)
calib_model = build_model(cfg).to(device)

criterion_calib = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_val], device=device)
)
optimizer_calib = torch.optim.Adam(calib_model.parameters(), lr=LR)

calib_history = []
print(f'Calibration: {CALIB_EPOCHS} epochs on {CALIB_SUBSET*100:.0f}% of real PaySim\n')

for epoch in range(1, CALIB_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(calib_model, calib_loader, optimizer_calib, criterion_calib, device)
    vl_loss, vl_acc = eval_epoch(calib_model, val_loader, criterion_calib, device)
    elapsed = time.time() - t0
    print(f'Epoch {epoch}/{CALIB_EPOCHS}  '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4%}  '
          f'val_loss={vl_loss:.4f}  val_acc={vl_acc:.4%}  ({elapsed:.1f}s)')
    calib_history.append({
        'epoch': epoch, 'train_loss': tr_loss, 'train_accuracy': tr_acc,
        'val_loss': vl_loss, 'val_accuracy': vl_acc, 'elapsed_s': round(elapsed, 2)
    })

os.makedirs('results', exist_ok=True)
with open('results/calibration_run_01.json', 'w') as f:
    json.dump({
        'run': 'calibration_run_01',
        'data': 'PaySim real (ealaxi/paysim1)',
        'subset_fraction': CALIB_SUBSET,
        'epochs': CALIB_EPOCHS,
        'pos_weight': pos_weight_val,
        'final_val_accuracy': calib_history[-1]['val_accuracy'],
        'final_val_loss': calib_history[-1]['val_loss'],
        'history': calib_history,
    }, f, indent=2)
print('\nCalibration complete. Saved: results/calibration_run_01.json')

---
# DAY 4 — Full Training (100% data, 10 epochs)

In [ ]:
EPOCHS = cfg['training']['epochs']  # 10

torch.manual_seed(SEED)
full_model = build_model(cfg).to(device)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_val], device=device)
)
optimizer = torch.optim.Adam(full_model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=2, factor=0.5, verbose=True
)

history = []
best_val_acc = 0.0
os.makedirs('models', exist_ok=True)

print(f'Full training: {EPOCHS} epochs on {len(X_train):,} samples (real PaySim)\n')

In [ ]:
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(full_model, train_loader, optimizer, criterion, device)
    vl_loss, vl_acc = eval_epoch(full_model, val_loader, criterion, device)
    scheduler.step(vl_loss)
    elapsed = time.time() - t0

    print(f'Epoch {epoch:02d}/{EPOCHS}  '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4%}  '
          f'val_loss={vl_loss:.4f}  val_acc={vl_acc:.4%}  ({elapsed:.1f}s)')

    history.append({
        'epoch': epoch, 'train_loss': tr_loss, 'train_accuracy': tr_acc,
        'val_loss': vl_loss, 'val_accuracy': vl_acc, 'elapsed_s': round(elapsed, 2)
    })

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(full_model.state_dict(), 'models/lstm_checkpoint_best.pt')
        print(f'  → New best val_acc={vl_acc:.4%} — checkpoint saved')

    if epoch == 5 and tr_acc < 0.90:
        print('  WARNING: Accuracy < 90% at epoch 5 — consider reducing batch_size to 256 or adding weight_decay')

torch.save(full_model.state_dict(), 'models/lstm_final.pt')
with open('results/training_history.json', 'w') as f:
    json.dump({'epochs': EPOCHS, 'pos_weight': pos_weight_val, 'history': history}, f, indent=2)

print(f'\nDone. Best val_acc: {best_val_acc:.4%}')
print('Saved: models/lstm_final.pt  models/lstm_checkpoint_best.pt  results/training_history.json')

## Training Curves Plot

In [ ]:
import matplotlib.pyplot as plt

epochs_x   = [h['epoch'] for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss   = [h['val_loss'] for h in history]
train_acc  = [h['train_accuracy'] for h in history]
val_acc    = [h['val_accuracy'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LSTM Training Curves — Meridian Sentinel', fontsize=14, fontweight='bold')

ax1.plot(epochs_x, train_loss, 'b-o', label='Train Loss')
ax1.plot(epochs_x, val_loss, 'orange', linestyle='-', marker='s', label='Val Loss')
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCEWithLogitsLoss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_x, train_acc, 'b-o', label='Train Accuracy')
ax2.plot(epochs_x, val_acc, 'orange', linestyle='-', marker='s', label='Val Accuracy')
ax2.axhline(0.95, color='red', linestyle='--', linewidth=1.5, label='95% Target')
ax2.axhline(0.9855, color='green', linestyle='--', linewidth=1.5, label='98.55% Final Target')
ax2.set_title('Accuracy per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1.05)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/training_curves.png')

## End of Day 4

**Save your model to Drive** (`.pt` files are gitignored):
```python
from google.colab import drive
drive.mount('/content/drive')
!cp models/lstm_checkpoint_best.pt /content/drive/MyDrive/meridian/models/
!cp models/lstm_final.pt /content/drive/MyDrive/meridian/models/
```

**Commit to `dev` (these files are NOT gitignored):**
- `results/calibration_run_01.json`
- `results/training_history.json`
- `results/figures/training_curves.png`
- `src/models/lstm_model.py`
- `config/model_config.yaml`

Proceed to `03_evaluation.ipynb` for Day 5.